<a href="https://colab.research.google.com/github/Saahar2001/Amanah-/blob/main/Amanah%20AI%20model/notebooks/model_training_v0_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semantic Integrity Model — Training and Validation

This notebook runs the measured training, validation, packaging, and deployment pipeline for the semantic-integrity classifier.

It uses a persistent Google Drive cache so verified sources, deterministic dataset splits, trained checkpoints, evaluation artifacts, and packaged releases can be reused across Colab sessions. Interrupted training resumes from the last completed epoch saved to the persistent cache.


In [2]:
# 0. GPU preflight — run this first
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        '\nAMANAH v0.2 training needs a GPU for the first training run.\n\n'
        'In Google Colab choose:\n'
        'Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save\n\n'
        'Then run the notebook again from the top.\n'
        'Your persistent Google Drive cache is reused, so completed cached work is not lost.\n'
        'Do not switch to CPU-only training for the measured release; it would be impractically slow.'
    )
print('GPU:', torch.cuda.get_device_name(0))
print('GPU PRECHECK: PASS')


Python: 3.13.15
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU PRECHECK: PASS


In [3]:
# 1. Initialize the project workspace from a repository archive
from pathlib import Path
from urllib.request import Request, urlopen
import io, json, os, platform, shutil, subprocess, sys, zipfile
NOTEBOOK_RELEASE = "2026.10.04-v0.2-r4"
print("Notebook release:", NOTEBOOK_RELEASE)


REPOSITORY_ID = 1407563428
headers = {"User-Agent": "semantic-integrity-training", "Accept": "application/vnd.github+json"}

metadata_request = Request(f"https://api.github.com/repositories/{REPOSITORY_ID}", headers=headers)
with urlopen(metadata_request, timeout=30) as response:
    repository_metadata = json.load(response)

default_branch = "main"
commit_request = Request(
    f"https://api.github.com/repositories/{REPOSITORY_ID}/commits/{default_branch}",
    headers=headers
)
with urlopen(commit_request, timeout=30) as response:
    commit_metadata = json.load(response)
commit_sha = commit_metadata["sha"]

archive_request = Request(
    f"https://api.github.com/repositories/{REPOSITORY_ID}/zipball/{commit_sha}",
    headers=headers
)
with urlopen(archive_request, timeout=120) as response:
    archive_bytes = response.read()

workspace = Path("/content/semantic_integrity_project")
extract_root = Path("/content/semantic_integrity_source")
for path in (workspace, extract_root):
    if path.exists():
        shutil.rmtree(path)
extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    archive.extractall(extract_root)

roots = [path for path in extract_root.iterdir() if path.is_dir()]
if len(roots) != 1:
    raise RuntimeError(f"Unexpected repository archive layout: {len(roots)} root directories")
shutil.move(str(roots[0]), str(workspace))
project_root = workspace / "Amanah AI model"
if not project_root.exists():
    raise RuntimeError(f"Submission model directory not found: {project_root}")
os.chdir(project_root)

print("Repository ID:", REPOSITORY_ID)
print("Commit:", commit_sha[:12])
print("Workspace:", project_root)
print("Python:", platform.python_version())


Notebook release: 2026.10.04-v0.2-r2
Repository ID: 1391259908
Commit: 832863f594e1
Workspace: /content/semantic_integrity_project
Python: 3.13.15


In [ ]:
# 2. Prepare the Colab environment and run lightweight quality gates
import importlib.util
import importlib.metadata
import time

required_packages = {
    "pydantic": "pydantic>=2.8,<3",
    "fastapi": "fastapi>=0.115,<1",
    "sklearn": "scikit-learn>=1.5",
    "transformers": "transformers>=4.45,<5",
    "sentencepiece": "sentencepiece>=0.2,<1",
    "httpx": "httpx>=0.27",
    "requests": "requests>=2.32",
    "huggingface_hub": "huggingface_hub>=0.35,<1"
}

install_specs = []
for module, spec in required_packages.items():
    if importlib.util.find_spec(module) is None:
        install_specs.append(spec)
try:
    transformers_major = int(importlib.metadata.version("transformers").split(".", 1)[0])
    if transformers_major >= 5:
        install_specs.append("transformers>=4.45,<5")
except importlib.metadata.PackageNotFoundError:
    pass

install_specs = list(dict.fromkeys(install_specs))
if install_specs:
    print("Installing compatible runtime packages:", install_specs)
    commands = [
        [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "--prefer-binary", "--break-system-packages", *install_specs],
        [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "--prefer-binary", *install_specs]
    ]
    last_error = None
    for index, command in enumerate(commands):
        try:
            subprocess.run(command, check=True)
            last_error = None
            break
        except subprocess.CalledProcessError as exc:
            last_error = exc
            if index == 0:
                print("Retrying dependency installation with Colab-compatible fallback...")
                time.sleep(3)
    if last_error is not None:
        raise RuntimeError("Dependency installation failed in the current Colab runtime.") from last_error
else:
    print("Required packages are already compatible; no pip changes needed.")

subprocess.run([sys.executable, "-m", "training.train", "--smoke"], check=True)
subprocess.run(
    [sys.executable, "-m", "compileall", "-q",
     "amanah_engine", "api", "data", "training", "scripts", "deployment"],
    check=True
)

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU session detected. This is fine when a completed checkpoint is available in persistent cache.")


Installing compatible runtime packages: ['transformers>=4.45,<5']


In [ ]:
# 3. Configure persistent cache and resumable training state
from datetime import datetime, timezone
import hashlib

MODEL_PIPELINE_VERSION = "v0.1"

TRAINING_CONFIG = {
    "pipeline_version": MODEL_PIPELINE_VERSION,
    "base_model": "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    "seed": 42,
    "epochs": 5,
    "batch_size": 4,
    "gradient_accumulation": 4,
    "learning_rate": 2e-5,
    "max_length": 512,
    "patience": 2,
}

signature_files = [
    "training/modeling.py",
    "data/models.py",
    "data/mutations.py",
    "data/validators.py",
    "data/split_dataset.py",
    "scripts/prepare_amanah_sd.py",
]
signature = hashlib.sha256(json.dumps(TRAINING_CONFIG, sort_keys=True).encode("utf-8"))
for relative_path in signature_files:
    signature.update(Path(relative_path).read_bytes())
TRAINING_SIGNATURE = signature.hexdigest()[:16]

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CACHE_ROOT = Path("/content/drive/MyDrive/semantic_integrity_model_cache")
    CACHE_IS_PERSISTENT = True
except Exception as exc:
    CACHE_ROOT = Path("/content/semantic_integrity_model_cache")
    CACHE_IS_PERSISTENT = False
    print("Google Drive cache unavailable; using this Colab session only:", type(exc).__name__)

SOURCE_CACHE = CACHE_ROOT / "sources"
RUN_CACHE = CACHE_ROOT / "runs" / TRAINING_SIGNATURE
SOURCE_CACHE.mkdir(parents=True, exist_ok=True)
RUN_CACHE.mkdir(parents=True, exist_ok=True)

def restore_file(cache_path: Path, local_path: Path) -> bool:
    if not cache_path.exists():
        return False
    local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(cache_path, local_path)
    return True

def persist_file(local_path: Path, cache_path: Path) -> None:
    if not local_path.exists():
        return
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    temp = cache_path.with_name(cache_path.name + ".tmp")
    shutil.copy2(local_path, temp)
    temp.replace(cache_path)

def restore_tree(cache_path: Path, local_path: Path) -> bool:
    if not cache_path.exists():
        return False
    if local_path.exists():
        shutil.rmtree(local_path)
    shutil.copytree(cache_path, local_path)
    return True

def persist_tree(local_path: Path, cache_path: Path) -> None:
    if not local_path.exists():
        return
    temp = cache_path.with_name(cache_path.name + ".tmp")
    if temp.exists():
        shutil.rmtree(temp)
    shutil.copytree(local_path, temp)
    if cache_path.exists():
        shutil.rmtree(cache_path)
    temp.replace(cache_path)

manifest = {
    "training_signature": TRAINING_SIGNATURE,
    "training_config": TRAINING_CONFIG,
    "source_commit": commit_sha,
    "updated_at": datetime.now(timezone.utc).isoformat(),
    "persistent": CACHE_IS_PERSISTENT,
}
(RUN_CACHE / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Training signature:", TRAINING_SIGNATURE)
print("Persistent cache:", CACHE_ROOT)
print("Cache persistence:", "Google Drive" if CACHE_IS_PERSISTENT else "current Colab session only")


In [ ]:
# 4. Restore or retrieve the trusted reference bundle
Path("data/private").mkdir(parents=True, exist_ok=True)
Path("artifacts/semantic_dataset").mkdir(parents=True, exist_ok=True)
Path("checkpoints").mkdir(parents=True, exist_ok=True)

bundle_path = Path("data/private/reference_bundle.json")
store_path = Path("data/reference_store.json")
bundle_cache = SOURCE_CACHE / "reference_bundle.json"
store_cache = SOURCE_CACHE / "reference_store.json"

restored_sources = restore_file(bundle_cache, bundle_path) and restore_file(store_cache, store_path)
source_cache_valid = False
if restored_sources:
    try:
        with bundle_path.open(encoding="utf-8") as handle:
            cached_bundle = json.load(handle)
        source_cache_valid = (
            len(cached_bundle.get("ayahs", [])) == 6236
            and all(len(row.get("references", [])) == 3 for row in cached_bundle["ayahs"])
        )
    except Exception:
        source_cache_valid = False

if source_cache_valid:
    print("Source cache: HIT — skipping external source retrieval.")
else:
    print("Source cache: MISS — retrieving verified sources once.")
    subprocess.run([
        sys.executable, "-m", "scripts.fetch_verified_sources",
        "--output", str(bundle_path),
        "--runtime-store", str(store_path),
        "--timeout", "120"
    ], check=True)
    persist_file(bundle_path, bundle_cache)
    persist_file(store_path, store_cache)

with bundle_path.open(encoding="utf-8") as handle:
    bundle = json.load(handle)
assert len(bundle["ayahs"]) == 6236
assert all(len(row["references"]) == 3 for row in bundle["ayahs"])
print("Verified canonical ayat:", len(bundle["ayahs"]))


In [ ]:
# 5. Restore or build the training dataset, then enforce QA
dataset_dir = Path("artifacts/semantic_dataset")
dataset_cache = RUN_CACHE / "semantic_dataset"
required_dataset_files = ["train.jsonl", "validation.jsonl", "test.jsonl", "dataset_summary.json"]

cache_ready = dataset_cache.exists() and all((dataset_cache / name).exists() for name in required_dataset_files)
if cache_ready:
    restore_tree(dataset_cache, dataset_dir)
    print("Dataset cache: HIT — skipping dataset regeneration.")
else:
    print("Dataset cache: MISS — building deterministic dataset.")
    subprocess.run([
        sys.executable, "-m", "scripts.prepare_amanah_sd",
        "data/private/reference_bundle.json",
        str(dataset_dir),
        "--seed", str(TRAINING_CONFIG["seed"])
    ], check=True)

qa_command = [
    sys.executable, "-m", "scripts.qa_dataset",
    "--reference-bundle", "data/private/reference_bundle.json",
    "--split-dir", str(dataset_dir),
    "--output", str(dataset_dir / "qa_report.json")
]
qa_process = subprocess.run(qa_command, text=True, capture_output=True)
if qa_process.stdout.strip():
    print(qa_process.stdout)
if qa_process.stderr.strip():
    print(qa_process.stderr)

qa_path = dataset_dir / "qa_report.json"
if not qa_path.exists():
    raise RuntimeError("Dataset QA did not produce qa_report.json. See stderr above.")
with qa_path.open(encoding="utf-8") as handle:
    qa = json.load(handle)
if qa_process.returncode != 0 or qa.get("status") != "ok":
    raise RuntimeError(f"Dataset QA failed: {qa.get('failures', [])}")
assert qa["split_ayah_leakage"] == []

if not cache_ready:
    persist_tree(dataset_dir, dataset_cache)
print("Dataset QA: PASS")


In [ ]:
# 6. Restore, resume, or run measured fine-tuning
checkpoint_dir = RUN_CACHE / "checkpoint"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
complete_marker = checkpoint_dir / "training_complete.json"
required_checkpoint_files = ["model.pt", "amanah_config.json", "training_complete.json"]

training_complete = complete_marker.exists() and all((checkpoint_dir / name).exists() for name in required_checkpoint_files)
if training_complete:
    print("Checkpoint cache: HIT — measured fine-tuning skipped. GPU is not required.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "Training is not complete in persistent cache. In Colab choose Runtime → Change runtime type → Hardware accelerator → T4 GPU, then rerun from the top. "
            "the notebook will resume from the last saved epoch if available."
        )

    # Only download/validate the tokenizer when training is actually required.
    preflight = subprocess.run(
        [sys.executable, "-c",
         "from transformers import AutoTokenizer; "
         "AutoTokenizer.from_pretrained('MoritzLaurer/mDeBERTa-v3-base-mnli-xnli'); "
         "print('Tokenizer preflight: PASS')"],
        text=True, capture_output=True
    )
    if preflight.stdout.strip():
        print(preflight.stdout)
    if preflight.returncode != 0:
        print(preflight.stderr)
        raise RuntimeError("Tokenizer preflight failed; see the detailed error above.")

    resume_available = (checkpoint_dir / "resume_model.pt").exists() and (checkpoint_dir / "resume_state.json").exists()
    if resume_available:
        print("Checkpoint cache: PARTIAL — resuming from the last completed epoch.")
    else:
        print("Checkpoint cache: MISS — starting measured fine-tuning.")

    train_command = [
        sys.executable, "-m", "training.train",
        "--train", str(dataset_dir / "train.jsonl"),
        "--validation", str(dataset_dir / "validation.jsonl"),
        "--output-dir", str(checkpoint_dir),
        "--epochs", str(TRAINING_CONFIG["epochs"]),
        "--batch-size", str(TRAINING_CONFIG["batch_size"]),
        "--gradient-accumulation", str(TRAINING_CONFIG["gradient_accumulation"]),
        "--learning-rate", str(TRAINING_CONFIG["learning_rate"]),
        "--max-length", str(TRAINING_CONFIG["max_length"]),
        "--patience", str(TRAINING_CONFIG["patience"]),
    ]
    if resume_available:
        train_command.extend(["--resume", str(checkpoint_dir)])

    train_process = subprocess.run(train_command, text=True, capture_output=True)
    if train_process.stdout.strip():
        print(train_process.stdout)
    if train_process.returncode != 0:
        print("TRAINING STDERR:\n", train_process.stderr)
        print("Partial epoch checkpoint, if available, remains saved in:", checkpoint_dir)
        raise RuntimeError(f"Fine-tuning failed with exit code {train_process.returncode}. See TRAINING STDERR above.")
    print("Fine-tuning: COMPLETE and persisted.")

with complete_marker.open(encoding="utf-8") as handle:
    training_state = json.load(handle)
print("Training state:", json.dumps(training_state, indent=2))


In [ ]:
# 7. Restore or run threshold calibration and held-out evaluation
evaluation_dir = Path("artifacts/evaluation")
evaluation_cache = RUN_CACHE / "evaluation"
cached_thresholds = checkpoint_dir / "thresholds.json"
cached_metrics = evaluation_cache / "metrics.json"

if cached_thresholds.exists() and cached_metrics.exists():
    restore_tree(evaluation_cache, evaluation_dir)
    print("Evaluation cache: HIT — calibration and held-out evaluation skipped.")
else:
    print("Evaluation cache: MISS — calibrating thresholds and evaluating held-out test split.")
    print("Progress will stream live below; GPU inference should report device=cuda.")

    calibration_command = [
        sys.executable, "-m", "training.calibrate_thresholds",
        "--checkpoint", str(checkpoint_dir),
        "--validation", str(dataset_dir / "validation.jsonl"),
    ]
    calibration = subprocess.run(calibration_command)
    if calibration.returncode != 0:
        raise RuntimeError(
            f"Threshold calibration failed with exit code {calibration.returncode}. "
            "The detailed root cause is printed above; the trained checkpoint remains cached."
        )

    evaluation_command = [
        sys.executable, "-m", "training.evaluate",
        "--checkpoint", str(checkpoint_dir),
        "--test", str(dataset_dir / "test.jsonl"),
        "--output", str(evaluation_dir),
    ]
    evaluation = subprocess.run(evaluation_command)
    if evaluation.returncode != 0:
        raise RuntimeError(
            f"Held-out evaluation failed with exit code {evaluation.returncode}. "
            "The detailed root cause is printed above; the trained checkpoint remains cached."
        )
    persist_tree(evaluation_dir, evaluation_cache)

with (evaluation_dir / "metrics.json").open(encoding="utf-8") as handle:
    metrics = json.load(handle)
assert metrics.get("status") == "ok", metrics
for key in ["macro_f1", "critical_drift_recall", "false_safe_rate", "severity_macro_f1", "coverage", "abstention_rate", "critical_coverage"]:
    assert 0.0 <= metrics[key] <= 1.0
print(json.dumps(metrics, indent=2))
print(
    "Evaluation note: overlength inputs are never silently truncated at runtime; "
    "they are counted as fail-closed ABSTAIN and reported through coverage/abstention metrics."
)


In [ ]:
# 8. Restore or package the model, then validate the inference contract once
model_repository = Path("artifacts/model_repository")
model_repository_cache = RUN_CACHE / "model_repository"
inference_smoke_marker = RUN_CACHE / "inference_smoke.json"
required_files = [
    "handler.py", "model.pt", "amanah_config.json", "thresholds.json",
    "reference_store.json", "requirements.txt"
]

package_cache_ready = model_repository_cache.exists() and all((model_repository_cache / name).exists() for name in required_files)
if package_cache_ready:
    restore_tree(model_repository_cache, model_repository)
    print("Model package cache: HIT — packaging skipped.")
else:
    subprocess.run([
        sys.executable, "-m", "scripts.package_hf_model",
        "--checkpoint", str(checkpoint_dir),
        "--reference-store", "data/reference_store.json",
        "--output", str(model_repository)
    ], check=True)
    missing = [name for name in required_files if not (model_repository / name).exists()]
    assert not missing, f"Missing deployment files: {missing}"
    persist_tree(model_repository, model_repository_cache)

missing = [name for name in required_files if not (model_repository / name).exists()]
assert not missing, f"Missing deployment files: {missing}"

if inference_smoke_marker.exists():
    print("Inference smoke cache: HIT — local full-model smoke skipped.")
    response = json.loads(inference_smoke_marker.read_text(encoding="utf-8"))
else:
    sys.path.insert(0, str(model_repository.resolve()))
    from handler import EndpointHandler
    handler = EndpointHandler(str(model_repository))
    with (model_repository / "reference_store.json").open(encoding="utf-8") as handle:
        reference_store = json.load(handle)
    row = reference_store["ayahs"][0]
    response = handler({"inputs": {
        "source_type": "quran",
        "source_ar": row["source_ar"],
        "candidate_en": row["references_en"][0],
        "ayah_id": row["ayah_id"]
    }})
    inference_smoke_marker.write_text(json.dumps(response, ensure_ascii=False, indent=2), encoding="utf-8")

assert response["decision"] in {"PASS", "REVIEW", "CRITICAL", "ABSTAIN"}
assert response["reference_status"] == "verified"
print(json.dumps(response, ensure_ascii=False, indent=2))


In [ ]:
# 9. Write and persist the measured validation report and release archive
report = {
    "dataset_summary": json.load(open(dataset_dir / "dataset_summary.json", encoding="utf-8")),
    "dataset_qa": qa,
    "evaluation": metrics,
    "training_state": training_state,
    "training_signature": TRAINING_SIGNATURE,
    "model_repository_url": None,
    "inference_endpoint_url": None,
    "deployment_status": "not_started"
}

report_path = Path("artifacts/VALIDATION_REPORT.json")
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
release_path = Path("/content/SEMANTIC_INTEGRITY_MODEL_RELEASE.zip")
if release_path.exists():
    release_path.unlink()
shutil.make_archive(str(release_path.with_suffix("")), "zip", str(model_repository))

persist_file(report_path, RUN_CACHE / "VALIDATION_REPORT.json")
persist_file(release_path, RUN_CACHE / "SEMANTIC_INTEGRITY_MODEL_RELEASE.zip")

print(json.dumps(report, ensure_ascii=False, indent=2))
print("Release archive:", release_path)
print("Persistent release copy:", RUN_CACHE / "SEMANTIC_INTEGRITY_MODEL_RELEASE.zip")


In [ ]:
# 10. Publish the validated package to Hugging Face when write access is available
from huggingface_hub import HfApi, create_repo
from huggingface_hub.errors import HfHubHTTPError

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    print("HF_TOKEN is not configured in Colab Secrets.")
    print("Create a Hugging Face User Access Token with WRITE permission, save it as HF_TOKEN, then rerun only this cell.")
    report["deployment_status"] = "package_ready_hf_write_token_required"
else:
    api = HfApi(token=hf_token)
    try:
        identity = api.whoami()
    except Exception as exc:
        raise RuntimeError("HF_TOKEN could not authenticate to Hugging Face.") from exc

    namespace = identity.get("name")
    if not namespace:
        raise RuntimeError("Hugging Face account namespace could not be resolved from this token.")
    print("Hugging Face authentication: PASS")
    print("Hugging Face account:", namespace)
    model_repo_id = f"{namespace}/semantic-integrity-v0-1"

    try:
        create_repo(
            repo_id=model_repo_id,
            repo_type="model",
            private=True,
            exist_ok=True,
            token=hf_token
        )
        print("Hugging Face write permission: PASS")

        api.upload_folder(
            repo_id=model_repo_id,
            repo_type="model",
            folder_path=str(model_repository),
            token=hf_token,
            commit_message="Publish validated v0.1 checkpoint"
        )
        api.upload_file(
            path_or_fileobj=str(report_path),
            path_in_repo="VALIDATION_REPORT.json",
            repo_id=model_repo_id,
            repo_type="model",
            token=hf_token,
            commit_message="Add measured validation report"
        )

        report["model_repository_url"] = f"https://huggingface.co/{model_repo_id}"
        report["deployment_status"] = "model_uploaded"
        print("Hugging Face upload: PASS")
        print("Model repository:", report["model_repository_url"])
    except HfHubHTTPError as exc:
        status = getattr(getattr(exc, "response", None), "status_code", None)
        if status == 403:
            print("Hugging Face write permission: FAIL")
            print("The current HF_TOKEN authenticates but cannot create/write the model repository.")
            print("Replace HF_TOKEN with a User Access Token whose role is WRITE, then rerun only this cell.")
            report["deployment_status"] = "package_ready_hf_write_token_required"
        else:
            raise

report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
persist_file(report_path, RUN_CACHE / "VALIDATION_REPORT.json")
print("Training, evaluation, and packaging are complete.")
print("Deployment status:", report["deployment_status"])


In [ ]:
# 11. Final black-box acceptance test on held-out samples
from collections import OrderedDict

test_rows = [
    json.loads(line)
    for line in (dataset_dir / "test.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

wanted_labels = [
    "FAITHFUL", "NEGATION_FLIP", "OMISSION",
    "MODALITY_SHIFT", "QUANTIFIER_CHANGE", "CONDITION_LOSS"
]
selected = OrderedDict()
for row in test_rows:
    for label in row.get("labels", []):
        if label in wanted_labels and label not in selected:
            selected[label] = row
    if len(selected) == len(wanted_labels):
        break

sys.path.insert(0, str(model_repository.resolve()))
from handler import EndpointHandler
acceptance_handler = EndpointHandler(str(model_repository))

allowed_decisions = {"PASS", "REVIEW", "CRITICAL", "ABSTAIN"}
results = []
contract_failures = []
label_hits = 0

for expected_label, row in selected.items():
    result = acceptance_handler({"inputs": {
        "source_type": "quran",
        "source_ar": row["source_ar"],
        "candidate_en": row["candidate_en"],
        "ayah_id": row["ayah_id"]
    }})
    predicted_labels = [item.get("label") for item in result.get("drifts", [])]
    if expected_label == "FAITHFUL":
        label_match = result.get("decision") == "PASS" or "FAITHFUL" in predicted_labels
    else:
        label_match = expected_label in predicted_labels
    label_hits += int(label_match)

    contract_ok = (
        result.get("decision") in allowed_decisions
        and result.get("reference_status") == "verified"
        and isinstance(result.get("integrity_score"), (int, float))
        and isinstance(result.get("confidence"), (int, float))
        and isinstance(result.get("drifts"), list)
    )
    if not contract_ok:
        contract_failures.append(expected_label)

    results.append({
        "ayah_id": row["ayah_id"],
        "expected_label": expected_label,
        "decision": result.get("decision"),
        "severity": result.get("severity"),
        "confidence": result.get("confidence"),
        "integrity_score": result.get("integrity_score"),
        "predicted_labels": predicted_labels,
        "label_match": label_match,
        "contract_ok": contract_ok,
    })

acceptance = {
    "status": "PASS" if not contract_failures and metrics.get("status") == "ok" else "FAIL",
    "training_signature": TRAINING_SIGNATURE,
    "held_out_samples_tested": len(results),
    "contract_failures": contract_failures,
    "representative_label_hits": label_hits,
    "representative_label_total": len(results),
    "measured_evaluation": metrics,
    "samples": results,
}

acceptance_path = Path("artifacts/FINAL_ACCEPTANCE_REPORT.json")
acceptance_path.write_text(json.dumps(acceptance, ensure_ascii=False, indent=2), encoding="utf-8")
persist_file(acceptance_path, RUN_CACHE / "FINAL_ACCEPTANCE_REPORT.json")

print(json.dumps(acceptance, ensure_ascii=False, indent=2))
assert acceptance["status"] == "PASS", "Final model contract acceptance failed."
print("FINAL MODEL ACCEPTANCE: PASS")
print("Persistent report:", RUN_CACHE / "FINAL_ACCEPTANCE_REPORT.json")


## v0.2 release gate: live-audit failures
This cell is mandatory before publishing. It verifies the 35:28 agent/patient reversal and the 2:2 negation-equivalence regression reported by the compliance audit. A release must fail closed if the known bad 35:28 example is still accepted.


In [ ]:
from amanah_engine.semantic_rules import run_rules
from data.models import DriftLabel

ref_3528 = "Only those of His servants who have knowledge fear Allah."
bad_3528 = "Only Allah fears the knowledgeable among His servants. Indeed, Allah is Exalted in Might and Forgiving."
good_3528 = "Only those fear Allah, from among His servants, who have knowledge. Indeed, Allah is Exalted in Might and Forgiving."
bad_labels = {f.label.value for f in run_rules(ref_3528, bad_3528)}
good_labels = {f.label.value for f in run_rules(ref_3528, good_3528)}
assert "AGENCY_SHIFT" in bad_labels, bad_labels
assert "AGENCY_SHIFT" not in good_labels, good_labels

ref_22 = "This is the Book about which there is no doubt, a guidance for the mindful."
alt_22 = "This is the Book, beyond doubt, a guidance for the mindful."
assert "NEGATION_FLIP" not in {f.label.value for f in run_rules(ref_22, alt_22)}
print("v0.2 REGRESSION GATES: PASS")
print("Important: these are release gates, not a measured accuracy percentage.")
